<a href="https://colab.research.google.com/github/yernarbakatay/Human-vs-AI-generated-Text-Detection/blob/main/notebooks/Llama3_1_prompting_ai_vs_human.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Set up Ollama in Colab

To make `langchain_ollama.ChatOllama` work, we need an Ollama server running and the `llama3.1:8b` model available on it. Since Colab runs on a remote VM, we'll install Ollama directly into this environment.

In [1]:
# Dependencies
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install langchain_ollama

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (1,052 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently

Bringing the server up

In [2]:
# Start Ollama server in the background
import subprocess
import os

# Set the OLLAMA_HOST environment variable to allow connections from the notebook process
os.environ['OLLAMA_HOST'] = '0.0.0.0'

# Start Ollama server as a background process
process = subprocess.Popen(['ollama', 'serve'], preexec_fn=os.setsid, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Ollama server started in the background.")

Ollama server started in the background.


With Ollama now set up and the model pulled, the existing code in the subsequent cells that initializes and uses `ChatOllama` should work as expected.

In [3]:
import time
import requests

# Function to check if Ollama server is ready
def wait_for_ollama(max_retries=30, delay=5):
    for i in range(max_retries):
        try:
            response = requests.get('http://localhost:11434/')
            if response.status_code == 200:
                print("Ollama server is ready.")
                return True
        except requests.exceptions.ConnectionError:
            pass
        print(f"Waiting for Ollama server... ({i+1}/{max_retries})")
        time.sleep(delay)
    print("Ollama server did not become ready.")
    return False

# Wait for the Ollama server to be ready
if wait_for_ollama():
    # Pull the llama3.1:8b model
    print("Pulling llama3.1:8b model. This may take some time...")
    !ollama pull llama3.1:8b
    print("llama3.1:8b model pulled successfully.")
else:
    print("Cannot proceed: Ollama server is not running.")

Ollama server is ready.
Pulling llama3.1:8b model. This may take some time...

llama3.1:8b model pulled successfully.


In [4]:
!nvidia-smi

Sun Apr 19 01:54:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   28C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

Imports

In [5]:
import random
from langchain_ollama import ChatOllama
import pandas as pd

from tqdm import tqdm
import time
import re
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from sklearn.metrics import classification_report

Testing the model with a simplke example

In [6]:
# Initialize the model
# You can adjust 'temperature' for creativity vs. logic
model = ChatOllama(model="llama3.1:8b", temperature=0.7, top_p=0.9)

# Run a simple test
prompt = "Who are you?"
response = model.invoke(prompt)

print(response)

content="I’m a large language model. When you ask me a question or provide me with a prompt, I analyze what you say and generate a response that is relevant and accurate. I'm constantly learning and improving, so over time I'll be even better at assisting you. Is there anything I can help you with?" additional_kwargs={} response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-04-18T20:16:32.830181363Z', 'done': True, 'done_reason': 'stop', 'total_duration': 33198513250, 'load_duration': 32882901974, 'prompt_eval_count': 14, 'prompt_eval_duration': 9566847, 'eval_count': 64, 'eval_duration': 273834920, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'} id='lc_run--019da23c-628d-76e2-8a9b-f2bacd1d299f-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 14, 'output_tokens': 64, 'total_tokens': 78}


In [6]:
df = pd.read_csv("/content/drive/MyDrive/Colab/sampled_50_dataset.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4841 entries, 0 to 4840
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   ai_target     4841 non-null   int64 
 1   source_model  4841 non-null   object
 2   text30        4841 non-null   object
 3   text50        4841 non-null   object
 4   text100       4841 non-null   object
 5   text200       4841 non-null   object
dtypes: int64(1), object(5)
memory usage: 227.1+ KB


# Trying Vanilla Prompting on the 10% sample - 200 words length data

It is recommended for chat-tuned models like LLama 3.1 to use ***ChatPromptTemplate()*** instead of PromptTemplate.



In [ ]:
system_msg = "You are a classifier. Output ONLY '0' or '1'. No explanation."
user_msg = "Classify this text (0=Human, 1=AI):\n\n{text200}"

prompt = ChatPromptTemplate.from_messages([
    ("system", system_msg),
    ("user", user_msg)
])

chain = prompt | model

texts = df["text200"].tolist()
results = []
start = time.time()

batch_size = 16

for i in tqdm(range(0, len(texts), batch_size), desc="T4 Batch Processing"):
    batch_input = [{"text200": t} for t in texts[i : i + batch_size]]

    # .batch() is much faster on T4 than .invoke() in a loop
    responses = chain.batch(batch_input)

    for res in responses:
        # Clean the output string
        clean_content = res.content.strip()

        # Look for the last digit if the model was chatty
        matches = re.findall(r'\b[01]\b', clean_content)
        if matches:
            # We take the last match as the final answer
            results.append(int(matches[-1]))
        else:
            results.append(-1) # Error flag

df["ai_opinion_200_va"] = results
print(f"Total time: {time.time() - start:.2f}s")

T4 Batch Processing: 100%|██████████| 303/303 [27:03<00:00,  5.36s/it]

Total time: 1623.22s


In [ ]:
y_true = df['ai_target']
y_pred = df['ai_opinion_200_va']

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.50      0.95      0.65      2423
           1       0.38      0.03      0.06      2418

    accuracy                           0.49      4841
   macro avg       0.44      0.49      0.35      4841
weighted avg       0.44      0.49      0.35      4841



### Vanilla Prompting Experiment Results

| Dataset | Model | Temperature | Macro Avg F1-Score | Notes | Time |
| :--- | :--- | :--- | :--- | :--- | :--- |
| sampled_50_200 | llama3.1:8b | 0.7 | 0.43 | lower f1 score for AI class 0.25 compared to 0.61 | 26:59 |
| sampled_50_200 | llama3.1:8b | 0.2 | 0.35 | very bad AI detection with f1-score 0.06  | 27:03 |

One reason I can imagine it get's confused by AI text so much is because different LLMs have different patterns in writing and the agent cannot put all of them together. We can test this hypothesis by trying the other dataset which is not 50-50. Moreover we can adress this when doing the multi agent, by giving one agent a single source of AI generated text at a time.

Example of prompt after formatting:

In [ ]:
formatted_prompt = prompt.invoke({"text200": texts[1]})

# This returns the list of Message objects
for message in formatted_prompt.to_messages():
    print(f"Role: {message.type.upper()}")
    print(f"Content: {message.content}")
    print("-" * 20)

Role: SYSTEM
Content: You are a classifier. Output ONLY '0' or '1'. No explanation.
--------------------
Role: HUMAN
Content: Classify this text (0=Human, 1=AI):

Covid-19Guidance Track Covid-19 in Rolette County, North Dakota The New York TimesUpdatedMarch 26, 2024 Track Covid-19 in Rolette County, N.D. Daily Covid-19 admissions in the Rolette County area About the data Data is from the Centers for Disease Control and Prevention. Hospitalization data is a daily average of Covid-19 patients in hospital service areas that intersect with Rolette County, an area which may be larger than Rolette County itself. The number ofdaily hospital admissionsshows how many patients were admitted to hospitals for Covid and is one of the most reliably reported indicators of Covid’s impact on a community. About the data Data is from the Centers for Disease Control and Prevention. Hospitalization data is a daily average of Covid-19 patients in hospital service areas that intersect with Rolette County, an

# Principle-Based Prompting - Single Agent

### Principle Candidates Prompt - limited to 3-5

In [16]:
# --- PRINCIPLE GENERATION PROMPT ---
# gen_system_msg = """You are given the task to extract principles or important features which distinguish between text written by a human and text generated by an AI."""

# gen_user_msg = """Below are some examples of text.
# {formatted_examples}

# Can you analyze each statement and identify whether it was written by a human or an AI?
# Based on your analysis, can you extract principles or important features which distinguish between statements that are human-written and those that are AI-generated?"""

gen_system_msg = """You are an expert linguistics analyst specializing in AI forensics.
Your task is to extract core principles that distinguish between text written by a human and text generated by an AI (LLM)."""

gen_user_msg = """Below are examples of text. Analyze them carefully:
{formatted_examples}

Based on your analysis, extract a contrastive rubric. You must provide:
1. Two strict linguistic principles that indicate the text is HUMAN.
2. Two strict linguistic principles that indicate the text is AI.

Focus on structural differences, emotional nuance, and vocabulary. Output ONLY the 4 numbered principles."""


gen_prompt_template = ChatPromptTemplate.from_messages([
    ("system", gen_system_msg),
    ("human", gen_user_msg)
])

Formating the examples. Making sure we can feed both with and without label examples.

In [11]:
def format_examples_for_principles(df_subset, include_labels=True):
    """
    Formats rows from your dataframe into the 'Statement: <text>' format.
    """
    formatted_list = []
    for i, row in enumerate(df_subset.itertuples()):
        text = row.text200
        label_str = f" (Label: {'AI' if row.ai_target == 1 else 'Human'})" if include_labels else ""
        formatted_list.append(f"Statement {i+1}: {text}{label_str}")

    return "\n".join(formatted_list)

In [10]:
subset = df.head(5)
formatted_examples = format_examples_for_principles(subset, include_labels=True)

formatted_prompt = gen_prompt_template.invoke({"formatted_examples": formatted_examples})

print("--- RAW PROMPT INSPECTION ---")
for message in formatted_prompt.to_messages():
    # .type will show 'system' or 'human'
    print(f"Role: {message.type.upper()}")
    print(f"Content: {message.content}")
    print("-" * 30)

--- RAW PROMPT INSPECTION ---
Role: SYSTEM
Content: You are an expert linguistics analyst specializing in AI forensics.
Your task is to extract core principles that distinguish between text written by a human and text generated by an AI (LLM).
------------------------------
Role: HUMAN
Content: Below are examples of text. Analyze them carefully:
Statement 1: offer." However, travel experts emphasize the importance of staying informed and taking basic precautions. "While the risk of being personally affected by an attack remains low, it's crucial to be aware of your surroundings and follow local advice," said security consultant John Davies. "Avoid crowded areas, especially during peak hours, and stay vigilant for anything suspicious." The US State Department has issued travel advisories for several European countries, urging citizens to exercise increased caution and be aware of their surroundings. The advisories highlight the potential for terrorist attacks and recommend travelers reg

In [11]:
gen_model = ChatOllama(model="llama3.1:8b", temperature=0.8)
gen_chain = gen_prompt_template | gen_model

response = gen_chain.invoke({"formatted_examples": formatted_examples})
extracted_principles = response.content

print("--- EXTRACTED PRINCIPLES ---")
print(extracted_principles)

--- EXTRACTED PRINCIPLES ---
After analyzing the provided statements, I have identified the following 3-5 high-level principles or linguistic features that distinguish between AI-generated text and human-written text:

1. **Overuse of buzzwords and jargon**: AI-generated text tends to rely heavily on trendy terms and technical language to create a sense of authenticity and expertise. In contrast, human-written text often uses more straightforward and accessible language.
2. **Lack of nuance and subtlety**: AI-generated text frequently employs binary thinking, oversimplifying complex issues and omitting subtle shades of meaning that humans would naturally incorporate. This results in text that sounds "too good (or bad) to be true."
3. **Stilted sentence structure and awkward phrasing**: Human-written text often exhibits a more natural flow and cadence, whereas AI-generated text can sound stilted or forced, with awkwardly constructed sentences that lack the smoothness of human language.


### Classification prompt

In [17]:
# classifier_system_msg = """You are a high-precision classifier.
# Use the following principles to distinguish between Human and AI text:

# {extracted_principles}

# Task: Classify the user's text as 0 (Human) or 1 (AI).
# Constraint: Output ONLY the number. No prose."""
classifier_system_msg = """You are an objective, high-precision classifier.
Use the following rubric to weigh the user's text:

{extracted_principles}

Task: Determine if the text is Human (0) or AI (1).
Constraint: You may write exactly ONE short sentence of reasoning, followed by the final digit '0' or '1'."""

classifier_user_msg = "Text to classify:\n\n{text200}\n\nReasoning and final digit:"


classifier_prompt = ChatPromptTemplate.from_messages([
    ("system", classifier_system_msg),
    ("human", classifier_user_msg)
])

In [13]:
print("--- FULL CLASSIFICATION PROMPT ---")
formatted_classification_prompt = classifier_prompt.invoke({"extracted_principles": extracted_principles, "text200": df["text200"][100]})
for message in formatted_classification_prompt.to_messages():
    print(f"Role: {message.type.upper()}")
    print(f"Content: {message.content}")
    print("-" * 30)

--- FULL CLASSIFICATION PROMPT ---
Role: SYSTEM
Content: You are a high-precision classifier.
Use the following principles to distinguish between Human and AI text:

After analyzing the provided statements, I have identified the following 3-5 high-level principles or linguistic features that distinguish between AI-generated text and human-written text:

1. **Overuse of buzzwords and jargon**: AI-generated text tends to rely heavily on trendy terms and technical language to create a sense of authenticity and expertise. In contrast, human-written text often uses more straightforward and accessible language.
2. **Lack of nuance and subtlety**: AI-generated text frequently employs binary thinking, oversimplifying complex issues and omitting subtle shades of meaning that humans would naturally incorporate. This results in text that sounds "too good (or bad) to be true."
3. **Stilted sentence structure and awkward phrasing**: Human-written text often exhibits a more natural flow and cadence,

In [ ]:
classifier_model = ChatOllama(model="llama3.1:8b", temperature=0.0, top_p=0.9)

classification_chain = classifier_prompt | classifier_model
response = classification_chain.invoke({"extracted_principles": extracted_principles, "text200": df["text200"][100]})

print(response.content)
print(f"Actual label: {df['ai_target'][100]}")

0
Actual label: 1


### Experimental loop

Creating a balanced subsample of the sample to test improvements to the approach. Righ now AI detection is very bad.

In [7]:

human_subset = df[df['ai_target'] == 0].sample(n=100, random_state=42)
ai_subset = df[df['ai_target'] == 1].sample(n=100, random_state=42)

# 2. Combine them into a new testing dataframe
test_df = pd.concat([human_subset, ai_subset])

# 3. Shuffle the dataset and reset the index
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Verify the split
print(f"Test dataset ready! Total rows: {len(test_df)}")
print(test_df['ai_target'].value_counts())

Test dataset ready! Total rows: 200
ai_target
0    100
1    100
Name: count, dtype: int64


In [20]:

# 1. Setup our two distinct agents
gen_model = ChatOllama(model="llama3.1:8b", temperature=0.8)
classifier_model = ChatOllama(model="llama3.1:8b", temperature=0.0, top_p=0.9)

# 2. Chains
gen_chain = gen_prompt_template | gen_model
classification_chain = classifier_prompt | classifier_model

results = []
start = time.time()

human_indices = test_df[test_df['ai_target'] == 0].index.tolist()
ai_indices = test_df[test_df['ai_target'] == 1].index.tolist()

for i in tqdm(range(len(test_df)), desc="Processing Principles + Classification"):
    try:
        # --- PHASE 1: DYNAMIC PRINCIPLE GENERATION ---
        # 1. Grab 2 random Human examples (excluding current row)
        valid_human = [idx for idx in human_indices if idx != i]
        sample_human = random.sample(valid_human, 2)

        # 2. Grab 2 random AI examples (excluding current row)
        valid_ai = [idx for idx in ai_indices if idx != i]
        sample_ai = random.sample(valid_ai, 2)

        # Combine them and shuffle so the model doesn't just learn "first two are human"
        sample_indices = sample_human + sample_ai
        random.shuffle(sample_indices)

        subset = test_df.loc[sample_indices]
        formatted_examples = format_examples_for_principles(subset, include_labels=True)

        # Generate principles at Temp 0.8
        gen_response = gen_chain.invoke({"formatted_examples": formatted_examples})
        extracted_principles = gen_response.content.strip()

        # --- PHASE 2: CLASSIFICATION ---
        # Classify the current row using the newly generated principles at Temp 0.0
        target_text = test_df.at[i, "text200"]
        class_response = classification_chain.invoke({
            "extracted_principles": extracted_principles,
            "text200": target_text
        })

        # --- PHASE 3: PARSING ---
        clean_content = class_response.content.strip()
        matches = re.findall(r'\b[01]\b', clean_content)

        if matches:
            results.append(int(matches[-1]))
        else:
            results.append(-1)

    except Exception as e:
        print(f"Error on index {i}: {e}")
        results.append(-1)

test_df["ai_opinion_200_principle_based"] = results
print(f"Total time: {time.time() - start:.2f}s")

Processing Principles + Classification: 100%|██████████| 200/200 [05:09<00:00,  1.55s/it]

Total time: 309.86s


### Principle-Based Prompting Experiment Results

| Dataset | Model | Temperature | Macro Avg F1-Score | Notes | Time |
| :--- | :--- | :--- | :--- | :--- | :--- |
| sampled_50_200 | llama3.1:8b | 0.8, 0.0 | 0.22 | lower f1 score for AI class 0.04 compared to 0.61 | 2:42:19 |
| sampled_50_200_100 | llama3.1:8b | 0.8, 0.0 | 0.22 | lower f1 score for human now class 0.02 compared to 0.64 | 6:00|
| sampled_50_200_100 | llama3.1:8b | 0.8, 0.0 | 0.46 | adjustments in the promp instructions;0.58 - 0, 0.35 - 1 | 5:16|


The system cannot handle the classification. It either overpredicts one or the other class.

In [22]:
y_true = test_df['ai_target']
y_pred = test_df['ai_opinion_200_principle_based']

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.45      0.61      0.51       100
           1       0.38      0.24      0.29       100

    accuracy                           0.42       200
   macro avg       0.41      0.42      0.40       200
weighted avg       0.41      0.42      0.40       200



In [17]:
df['ai_opinion_200_principle_based'].value_counts()

,count
ai_opinion_200_principle_based,
0,4481
1,359
-1,1


# Principle-Based Prompting - Multi Agent Approach


Not worth it.

LESSON LEARNED: aim for explainable, tracable modelling, not black box; also, prompt engineering is bulshit